In [1]:
import json
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import MobileNetV2Config, MobileNetV2Model
from tqdm import tqdm
import sys

In [2]:

sys.path.append("/mnt/Personal/Projects/Autofocus/Code/Universal")

import pytorch_start # type: ignore
pytorch_start.activate_gpu(print_details=True)


                               ACTIVE GPU CONFIGURATION                              
+-------------+-----------------------+---------------+--------------+---------------+
|   Device ID | Name                  |   Memory (GB) | PCI Bus ID   | GFX Version   |
+=============+=======================+===============+==============+===============+
|           0 | AMD Radeon RX 9060 XT |         15.92 | Unknown      | Native        |
+-------------+-----------------------+---------------+--------------+---------------+

Configured GPU 0: _CudaDeviceProperties(name='AMD Radeon RX 9060 XT', major=12, minor=0, gcnArchName='gfx1200', total_memory=16304MB, multi_processor_count=16, uuid=38656231-6165-3864-3464-306530323065, pci_bus_id=3, pci_device_id=0, pci_domain_id=0, L2_cache_size=4MB)


In [3]:
dataset_type = "Test"
dataset_path = "/mnt/Velocity_Vault/Datasets/Autofocus/Dataset/"+dataset_type


label_path = dataset_path+"/label.mm"
patch_path = dataset_path+"/patch.mm"

meta_data_path = dataset_path+"/meta.txt"

with open(meta_data_path, "r") as f:
    meta_data = json.load(f)

In [4]:
class TestDataset(Dataset):
    def __init__(self, patches_path, labels_path, shape, label_offset=0, input_label_range=(0,-1)):
        
        self.patches = np.memmap(patches_path, dtype=np.uint8, mode='r', shape=shape)
        self.patches = self.patches[:,input_label_range[0]:input_label_range[1]]
        
        self.labels = np.memmap(labels_path, dtype=np.uint8, mode='r', shape=(shape[0],))

        mask = (self.labels >= 1) & (self.labels <= 40)
        self.indices = np.where(mask)[0]
        self.label_offset = label_offset

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]

        x = torch.from_numpy(self.patches[real_idx].copy())
        y = int(self.labels[real_idx]) - self.label_offset

        return x, torch.tensor(y, dtype=torch.long)

In [5]:


class FocusNet(nn.Module):
    def __init__(self, patch_size, num_channels):
        super().__init__()

        config = MobileNetV2Config(
            num_channels=num_channels,
            image_size=patch_size,
            depth_multiplier=0.5,
            output_stride=8,
            finegrained_output=True
        )

        self.backbone = MobileNetV2Model(config)

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.head = nn.Sequential(
            nn.Linear(1280, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, num_channels)
        )

    def forward(self, x):
        # x: (B,40,32,32)

        outputs = self.backbone(x)
        feat = outputs.last_hidden_state   # (B, C, H, W)

        feat = self.pool(feat).flatten(1)

        out = self.head(feat)

        return out

In [6]:
def load_model(model_path, device, patch_size, num_channels):
    model = FocusNet(patch_size=patch_size,num_channels=num_channels).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    return model

In [7]:


def evaluate(model, loader, device, acc_k=2):
    model.eval()

    total_mae = 0.0
    total_acck = 0.0
    
    avg_mae, avg_acck = 0.0, 0.0

    class_range = torch.arange(40, device=device).float()

    pbar = tqdm(
        loader,
        desc="Test",
        leave=True,
        dynamic_ncols=True,
        smoothing=0.1
    )

    with torch.no_grad():
        for i, (x, y) in enumerate(pbar, 1):

            x = x.to(device, non_blocking=True).float().div(127.5).sub(1.0)
            y = y.to(device, non_blocking=True)

            outputs = model(x)

            prob = torch.softmax(outputs, dim=1)
            pred_cont = (prob * class_range).sum(dim=1)

            pred = pred_cont.round().long()

            # ---- metrics ----
            mae = (pred_cont - y).abs().mean()
            total_mae += mae.item()

            acck_val = ((pred - y).abs() <= acc_k).float().mean()
            total_acck += acck_val.item()

            # ---- running averages ----
            avg_mae = total_mae / i
            avg_acck = total_acck / i

            # # ---- clean display ----
            # pbar.set_postfix_str(
            #     f"mae={avg_mae:.3f} | acc@{acc_k}={avg_acck:.3f}"
            # )

    print(f"\nResults:")
    print(f"MAE: {avg_mae:.4f}")
    print(f"Acc@{acc_k}: {avg_acck:.4f}")

    return avg_mae, avg_acck

In [8]:
# -----------------------------
# 5. Main
# -----------------------------
def run_test(
    patches_path,
    labels_path,
    shape,
    model_path,
    batch_size=1024,
    acc_k=2
):
    device = "cuda"

    dataset = TestDataset(patches_path, labels_path, shape, label_offset=1, input_label_range=(1,41))

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    print(f"Testing on {len(dataset)} samples...")

    model = load_model(model_path, device, patch_size=shape[-1], num_channels=40)

    evaluate(model, loader, device, acc_k)

In [9]:
# learning rate = 1e-4

model_path = "/mnt/Personal/Projects/Autofocus/Code/Universal/model_v1.pth"
run_test(patch_path,label_path,meta_data['patches']['shape'],model_path)

Testing on 60950 samples...


Test: 100%|██████████| 60/60 [00:04<00:00, 12.68it/s]


Results:
MAE: 2.0234
Acc@2: 0.7297


In [10]:
# learning rate = 3e-4

model_path = "/mnt/Personal/Projects/Autofocus/Code/Universal/model_v2.pth"
run_test(patch_path,label_path,meta_data['patches']['shape'],model_path)

Testing on 60950 samples...


Test: 100%|██████████| 60/60 [00:04<00:00, 13.87it/s]


Results:
MAE: 2.0405
Acc@2: 0.7350
